In [10]:
!pip install hmmlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 4.5 MB/s eta 0:00:00


In [13]:
import hmmlearn
from hmmlearn import hmm
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from matplotlib import pyplot as plt
from typing import List
import random
import re
import nltk
import warnings
from nltk.corpus import stopwords
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score,f1_score, roc_auc_score
nltk.download('stopwords')
warnings.filterwarnings('ignore')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
data=pd.read_csv('/content/NER dataset.csv',encoding='latin1')
data = data.fillna(method="ffill")
data = data.rename(columns={'Sentence #': 'sentence'})
data.head(5)

,sentence,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,Sentence: 1,of,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,have,VBP,O
4,Sentence: 1,marched,VBN,O


In [16]:
def pre_processing(text_column):
    text_column = text_column.str.lower()
    text_column = text_column.str.replace(r'\d+', 'NUM')
    stop_words = set(stopwords.words('english'))
    text_column = text_column.apply(lambda x: ' '.join([word for word in x.split() if word not in stop_words]))
    return text_column

In [19]:
data_pre_precessed = pre_processing(data.Word)
data_pre_precessed.head(20)

,Word
0,thousands
1,
2,demonstrators
3,
4,marched
5,
6,london
7,
8,protest
9,


In [21]:
data_processed = data
data_processed['Word'] = data_pre_precessed
data_processed = data_processed[(data_processed['Word'] != '') | (data_processed['Word'].isna())]
data_processed.head(20)

,sentence,Word,POS,Tag
0,Sentence: 1,thousands,NNS,O
2,Sentence: 1,demonstrators,NNS,O
4,Sentence: 1,marched,VBN,O
6,Sentence: 1,london,NNP,B-geo
8,Sentence: 1,protest,VB,O
10,Sentence: 1,war,NN,O
12,Sentence: 1,iraq,NNP,B-geo
14,Sentence: 1,demand,VB,O
16,Sentence: 1,withdrawal,NN,O
18,Sentence: 1,british,JJ,B-gpe


In [23]:
tags = list(set(data.POS.values))
words = list(set(data.Word.values))
words1 = list(set(data_processed.Word.values))
len(tags), len(words), len(words1)

(42, 31682, 31681)

In [24]:
y = data.POS
X = data.drop('POS', axis=1)
gs = GroupShuffleSplit(n_splits=2, test_size=.33, random_state=42)
train_ix, test_ix = next(gs.split(X, y, groups=data['sentence']))
data_train = data.loc[train_ix]
data_test = data.loc[test_ix]

In [25]:
data_train.head()

,sentence,Word,POS,Tag
24,Sentence: 2,families,NNS,O
25,Sentence: 2,,IN,O
26,Sentence: 2,soldiers,NNS,O
27,Sentence: 2,killed,VBN,O
28,Sentence: 2,,IN,O


In [26]:
data_test.head()

,sentence,Word,POS,Tag
0,Sentence: 1,thousands,NNS,O
1,Sentence: 1,,IN,O
2,Sentence: 1,demonstrators,NNS,O
3,Sentence: 1,,VBP,O
4,Sentence: 1,marched,VBN,O


In [27]:
y1 = data_processed.POS
X1 = data_processed.drop('POS', axis=1)
data_processed.reset_index(drop=True, inplace=True)
gs = GroupShuffleSplit(n_splits=2, test_size=.33, random_state=42)
train_ix1, test_ix1 = next(gs.split(X1, y1, groups=data_processed['sentence']))

data_train1 = data_processed.loc[train_ix1]
data_test1 = data_processed.loc[test_ix1]

In [28]:
data_train1.head()

,sentence,Word,POS,Tag
13,Sentence: 2,families,NNS,O
14,Sentence: 2,soldiers,NNS,O
15,Sentence: 2,killed,VBN,O
16,Sentence: 2,conflict,NN,O
17,Sentence: 2,joined,VBD,O


In [29]:
data_test1.head()

,sentence,Word,POS,Tag
0,Sentence: 1,thousands,NNS,O
1,Sentence: 1,demonstrators,NNS,O
2,Sentence: 1,marched,VBN,O
3,Sentence: 1,london,NNP,B-geo
4,Sentence: 1,protest,VB,O


In [30]:
dfupdate = data_train.sample(frac=.15, replace=False, random_state=42)
dfupdate.Word = 'UNKNOWN'
data_train.update(dfupdate)
words = list(set(data_train.Word.values))
word2id = {w: i for i, w in enumerate(words)}
tag2id = {t: i for i, t in enumerate(tags)}
id2tag = {i: t for i, t in enumerate(tags)}
len(tags), len(words)

(42, 24969)

In [31]:
count_tags = dict(data_train.POS.value_counts())
count_tags_to_words = data_train.groupby(['POS']).apply(
    lambda grp: grp.groupby('Word')['POS'].count().to_dict()).to_dict()
count_init_tags = dict(data_train.groupby('sentence').first().POS.value_counts())

count_tags_to_next_tags = np.zeros((len(tags), len(tags)), dtype=int)
sentences = list(data_train.sentence)
pos = list(data_train.POS)
for i in tqdm(range(len(sentences)), position=0, leave=True):
    if (i > 0) and (sentences[i] == sentences[i - 1]):
        prevtagid = tag2id[pos[i - 1]]
        nexttagid = tag2id[pos[i]]
        count_tags_to_next_tags[prevtagid][nexttagid] += 1

100%|██████████| 702936/702936 [00:00<00:00, 1093158.13it/s]


In [32]:
count_words = {}
for word in data_train.Word.values:
    count_words[word] = count_words.get(word, 0) + 1

count_word_transitions = {}
for sentence in data_train.groupby('sentence'):
    words = sentence[1]['Word'].values
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        if w1 not in count_word_transitions:
            count_word_transitions[w1] = {}
        count_word_transitions[w1][w2] = count_word_transitions[w1].get(w2, 0) + 1

word_transition_matrix = np.zeros((len(word2id)+1, len(word2id)+1))
sum_words_to_next_words = np.sum([count_word_transitions[w1][w2] for w1 in count_word_transitions for w2 in count_word_transitions[w1]])
for w1, w1id in word2id.items():
    for w2, w2id in word2id.items():
        word_transition_matrix[w1id][w2id] = count_word_transitions.get(w1, {}).get(w2, 0) / sum_words_to_next_words
print(word_transition_matrix.shape)

(24970, 24970)


In [33]:
def calculate_log_likelihood(sentence: List[str], word_transition_matrix) -> float:
    sentence_ids = [word2id.get(w, word2id['UNKNOWN']) for w in sentence]
    log_likelihood = np.log(word_transition_matrix[sentence_ids[0]][sentence_ids[1]])
    for i in range(1, len(sentence_ids) - 1):
        log_likelihood += np.log(word_transition_matrix[sentence_ids[i]][sentence_ids[i+1]] + 1e-10)
    return log_likelihood

In [34]:
calculate_log_likelihood(["This", "is", "a", "test", "sentence"], word_transition_matrix)

np.float64(-41.259970813020175)

In [37]:
model = hmm.MultinomialHMM(n_components=len(tags), algorithm='viterbi', random_state=42)
# Fit the model to the training data
# Prepare training data for fitting:
# X_train needs to be a sequence of observations. Here, we use word IDs.
# lengths_train is a list of lengths of sequences (sentences) in the training data.

X_train = []
for word in data_train.Word.values:
    X_train.append([word2id[word]])

lengths_train = []
count = 0
sentences_train = list(data_train.sentence)
for i in tqdm(range(len(sentences_train)), position=0, leave=True):
    if (i > 0) and (sentences_train[i] == sentences_train[i - 1]):
        count += 1
    elif i > 0:
        lengths_train.append(count)
        count = 1
    else:
        count = 1
lengths_train.append(count) # Append the last sentence length

model.fit(X_train, lengths_train)

https://github.com/hmmlearn/hmmlearn/issues/335
https://github.com/hmmlearn/hmmlearn/issues/340
100%|██████████| 702936/702936 [00:00<00:00, 2953416.01it/s]


MultinomialHMM(n_components=42,
               n_trials=array([22521,     0,  1767, ...,  1951,     0, 19260]),
               random_state=RandomState(MT19937) at 0x7E5F5A00BE40)

In [41]:
samples = []
for word in data_test.Word.values:
    samples.append([word2id.get(word, word2id['UNKNOWN'])]) # Handle unknown words

lengths = []
count = 0
sentences_test = list(data_test.sentence)
for i in tqdm(range(len(sentences_test)), position=0, leave=True):
    if (i > 0) and (sentences_test[i] == sentences_test[i - 1]):
        count += 1
    elif i > 0:
        lengths.append(count)
        count = 1
    else:
        count = 1
lengths.append(count) # Append the last sentence length


pos_predict = model.predict(samples, lengths)
pos_predict

100%|██████████| 345639/345639 [00:00<00:00, 2791128.70it/s]


array([25, 39, 22, ..., 29, 16, 12])

In [39]:
def viterbi(pi: np.array, a: np.array, b: np.array, obs: List) -> np.array:
    K = a.shape[0]
    T = len(obs)
    delta = np.zeros((T, K))
    psi = np.zeros((T, K))
    delta[0] = pi * b[:, obs[0]]
    for t in range(1, T):
        for j in range(K):
            delta[t, j] = np.max(delta[t-1] * a[:, j] * b[j, obs[t]])
            psi[t, j] = np.argmax(delta[t-1] * a[:, j])

    x = np.zeros(T, dtype=int)
    x[T-1] = np.argmax(delta[T-1])
    for t in range(T-2, -1, -1):
        x[t] = psi[t+1, x[t+1]]
    return x

In [42]:
hidden_states = ['healthy', 'sick']
observable_states = ['sleeping', 'eating', 'pooping']
observations = []
for i in range(100):
  observations.append(random.choice(observable_states))

In [48]:
import random
import numpy as np # Import numpy

hidden_states = ['Sunny', 'Cloudy', 'Rainy']
# observable_states = ['Hot', 'Mild', 'Cold', 'Windy', 'Foggy'] # Original states
observable_states = ['sleeping', 'eating', 'pooping'] # States matching emissionprob matrix
observable_map = {state: i for i, state in enumerate(observable_states)} # Map states to indices
observations =  []

for i in range(40):
  obs_index = random.randint(0, len(observable_states)-1)
  observations.append(obs_index)

In [49]:
hidden_state_sequence = viterbi(startprob, transmat, emissionprob, observations)

print("Observations:", observations)
print("Viterbi sequence:", hidden_state_sequence)

Observations: [0, 2, 2, 2, 0, 1, 2, 2, 1, 0, 0, 0, 1, 2, 0, 2, 1, 2, 1, 2, 2, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 2, 0, 1, 1, 1, 2, 2]
Viterbi sequence: [1 0 0 0 1 0 0 0 0 1 1 1 0 0 1 0 0 0 0 0 0 1 0 0 0 1 0 1 0 1 0 0 1 0 1 0 0
 0 0 0]


In [47]:
def baum_welch(observations, observations_vocab, n_hidden_states):
    def forward_probs(observations, observations_vocab, n_hidden_states, a_, b_) -> np.array:
        a_start = 1 / n_hidden_states
        alpha_ = np.zeros((n_hidden_states, len(observations)), dtype=float)
        alpha_[:, 0] = a_start
        for t in range(1, len(observations)):
          for j in range(n_hidden_states):
            calc = observations_vocab == observations[t]
            for i in range(n_hidden_states):
              alpha_[j, t] = sum(alpha_[i, t-1]*a_[i,j] * b_[j, np.where(calc)[0][0]] for i in range(n_hidden_states))
        return alpha_

    def backward_probs(observations, observations_vocab, n_hidden_states, a_, b_) -> np.array:
        beta_ = np.zeros((n_hidden_states, len(observations)), dtype=float)
        beta_[:, -1:] = 1
        for t in range(len(observations) -2, -1, -1):
          for i in range(n_hidden_states):
            calc2 = observations_vocab == observations[t+1]
            beta_[i,t] = sum(a_[i,j] * b_[j, np.where(calc2)[0][0]]*beta_[j, t+1] for j in range(n_hidden_states))
        return beta_

    def compute_gamma(alfa, beta, observations, vocab, n_samples, a_, b_) -> np.array:
        gamma_prob = np.multiply(alfa, beta) / sum(np.multiply(alfa, beta))
        return gamma_prob

    def compute_sigma(alfa, beta, observations, vocab, n_samples, a_, b_) -> np.array:
        sigma_prob = np.zeros((n_samples, len(observations) - 1, n_samples), dtype=float)
        denomenator = np.multiply(alfa, beta)
        for i in range(len(observations) - 1):
            for j in range(n_samples):
                for k in range(n_samples):
                    index_in_vocab = np.where(vocab == observations[i + 1])[0][0]
                    sigma_prob[j, i, k] = (alfa[j, i] * beta[k, i + 1] * a_[j, k] * b_[k, index_in_vocab]) / sum(
                        denomenator[:, j])
        return sigma_prob

    # initialize A ,B
    a = np.ones((n_hidden_states, n_hidden_states)) / n_hidden_states
    b = np.ones((n_hidden_states, len(observations_vocab))) / len(observations_vocab)
    for iter in tqdm(range(2000), position=0, leave=True):

        alfa_prob = forward_probs(observations, observations_vocab, n_hidden_states, a, b)  #
        beta_prob = backward_probs(observations, observations_vocab, n_hidden_states, a, b)  # , beta_val
        gamma_prob = compute_gamma(alfa_prob, beta_prob, observations, observations_vocab, n_hidden_states, a, b)
        sigma_prob = compute_sigma(alfa_prob, beta_prob, observations, observations_vocab, n_hidden_states, a, b)

        a_model = np.zeros((n_hidden_states, n_hidden_states))
        for j in range(n_hidden_states):  # calculate A-model
            for i in range(n_hidden_states):
                for t in range(len(observations) - 1):
                    a_model[j, i] = a_model[j, i] + sigma_prob[j, t, i]
                normalize_a = [sigma_prob[j, t_current, i_current] for t_current in range(len(observations) - 1) for
                               i_current in range(n_hidden_states)]
                normalize_a = sum(normalize_a)
                if normalize_a == 0:
                    a_model[j, i] = 0
                else:
                    a_model[j, i] = a_model[j, i] / normalize_a
        b_model = np.zeros((n_hidden_states, len(observations_vocab)))

        for j in range(n_hidden_states):
            for i in range(len(observations_vocab)):
                indices = [idx for idx, val in enumerate(observations) if val == observations_vocab[i]]
                numerator_b = sum(gamma_prob[j, indices])
                denominator_b = sum(gamma_prob[j, :])
                if denominator_b == 0:
                    b_model[j, i] = 0
                else:
                    b_model[j, i] = numerator_b / denominator_b

        a = a_model
        b = b_model
    return a, b

hidden_states = ['healthy', 'sick']
observable_states = ['sleeping', 'eating', 'pooping']
observable_map = {'sleeping': 0, 'eating': 1, 'pooping': 2}
observations = []
for i in range(40):
    observations.append(observable_map[random.choice(observable_states)])
A, B = baum_welch(observations=observations, observations_vocab=np.array(list(observable_map.values())),n_hidden_states=2)

100%|██████████| 2000/2000 [00:04<00:00, 477.75it/s]
